# 🏷️ Week 6: Part-of-Speech (POS) Tagging Bahasa Indonesia

## Tujuan Pembelajaran:
1. Memahami konsep **POS Tagging** — proses pemberian label kelas kata pada setiap token (kata benda, kata kerja, kata sifat, dll.)
2. Menggunakan **Stanza** (Stanford NLP) untuk POS tagging berbahasa Indonesia
3. Menerapkan deteksi bahasa (*language detection*) dengan **langdetect** sebelum melakukan tagging
4. Menganalisis distribusi POS tag dari seluruh dataset ulasan Honest Review
5. Menemukan kata (token) yang paling sering muncul berdasarkan kelas katanya

---
> 📂 **Dataset**: Honest Review (`cleandata.csv`) — kolom `text_final` (teks yang sudah dipreproses)
> File ini berada di folder `Week 3/` dalam workspace yang sama.
>
> ℹ️ **Stanza** digunakan karena mendukung bahasa Indonesia secara resmi dengan model POS tagging yang terlatih dari Stanford NLP.


## 1) Install Library

Install **Stanza** (Stanford NLP) dan **langdetect**. Stanza menyediakan model POS tagging berbahasa Indonesia yang terlatih secara resmi.


In [8]:
%pip install stanza langdetect


Note: you may need to restart the kernel to use updated packages.


In [9]:
import stanza

# Download model bahasa Indonesia (hanya perlu dijalankan sekali, ukuran ~500MB)
stanza.download('id', processors='tokenize,pos')
print("Model bahasa Indonesia berhasil didownload!")


c:\Users\mikba\Downloads\Documents\PBA\PBA\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-15 08:02:43 INFO: Downloaded file to C:\Users\mikba\AppData\Local\StanfordNLP\stanza\Cache\1.11.0\resources\resources.json
2026-04-15 08:02:43 WARNING: Language id package default expects mwt, which has been added
2026-04-15 08:02:43 INFO: Downloading these customized packages for language: id (Indonesian)...
| Processor       | Package    |
--------------------------------
| tokenize        | gsd        |
| mwt             | gsd        |
| pos             | gsd_charlm |
| pretrain        | conll17    |
| forward_charlm  | oscar2023  |
| backward_charlm | oscar2023  |

2026-04-15 08:02:43 INFO: File exists: C:\Users\mikba\AppData\Local\StanfordNLP\stanza\Cache\1.11.0\resources\id\tokenize\gsd.pt
2026-04-15 08:0

c:\Users\mikba\Downloads\Documents\PBA\PBA\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-15 08:02:43 INFO: Downloaded file to C:\Users\mikba\AppData\Local\StanfordNLP\stanza\Cache\1.11.0\resources\resources.json
2026-04-15 08:02:43 WARNING: Language id package default expects mwt, which has been added
2026-04-15 08:02:43 INFO: Downloading these customized packages for language: id (Indonesian)...
| Processor       | Package    |
--------------------------------
| tokenize        | gsd        |
| mwt             | gsd        |
| pos             | gsd_charlm |
| pretrain        | conll17    |
| forward_charlm  | oscar2023  |
| backward_charlm | oscar2023  |

2026-04-15 08:02:43 INFO: File exists: C:\Users\mikba\AppData\Local\StanfordNLP\stanza\Cache\1.11.0\resources\id\tokenize\gsd.pt
2026-04-15 08:0

Model bahasa Indonesia berhasil didownload!


## 2) Import Library

Import modul yang dibutuhkan untuk POS tagging dan deteksi bahasa.

In [10]:
import pandas as pd
import unicodedata
import re
from collections import defaultdict

import stanza
from langdetect import detect, LangDetectException

# Load pipeline Stanza bahasa Indonesia
nlp = stanza.Pipeline('id', processors='tokenize,pos', use_gpu=False)

print("Library berhasil diimport!")
print(f"Stanza version: {stanza.__version__}")


2026-04-15 08:02:48 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2026-04-15 08:02:49 INFO: Downloaded file to C:\Users\mikba\AppData\Local\StanfordNLP\stanza\Cache\1.11.0\resources\resources.json
2026-04-15 08:02:49 WARNING: Language id package default expects mwt, which has been added
2026-04-15 08:02:49 INFO: Loading these models for language: id (Indonesian):
| Processor | Package    |
--------------------------
| tokenize  | gsd        |
| mwt       | gsd        |
| pos       | gsd_charlm |

2026-04-15 08:02:49 INFO: Using device: cpu
2026-04-15 08:02:49 INFO: Loading: tokenize
2026-04-15 08:02:52 INFO: Loading: mwt
2026-04-15 08:02:52 INFO: Loading: pos
2026-04-15 08:02:54 INFO: Done loading processors!


2026-04-15 08:02:48 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2026-04-15 08:02:49 INFO: Downloaded file to C:\Users\mikba\AppData\Local\StanfordNLP\stanza\Cache\1.11.0\resources\resources.json
2026-04-15 08:02:49 WARNING: Language id package default expects mwt, which has been added
2026-04-15 08:02:49 INFO: Loading these models for language: id (Indonesian):
| Processor | Package    |
--------------------------
| tokenize  | gsd        |
| mwt       | gsd        |
| pos       | gsd_charlm |

2026-04-15 08:02:49 INFO: Using device: cpu
2026-04-15 08:02:49 INFO: Loading: tokenize
2026-04-15 08:02:52 INFO: Loading: mwt
2026-04-15 08:02:52 INFO: Loading: pos
2026-04-15 08:02:54 INFO: Done loading processors!


Library berhasil diimport!
Stanza version: 1.11.1


## 3) Load Dataset

Load dataset Honest Review dari `cleandata.csv`. Kolom `text_final` berisi teks ulasan yang sudah melalui preprocessing (tokenisasi, stopword removal, stemming).


In [11]:
file_path = '../Week 3/cleandata.csv'
df_text = pd.read_csv(file_path)

print(f"Jumlah data: {len(df_text)}")
print(f"Kolom: {list(df_text.columns)}")
df_text['text_final'].head()


Jumlah data: 39164
Kolom: ['score', 'content', 'text_final', 'reviewCreatedVersion']


0    kali kartu kredit cocok anak muda update alam ...
1    hasil layan cs nama sangat sopan jelas sangat ...
2                                           cepat aman
3                                              cs baik
4                                           jelas baik
Name: text_final, dtype: str

## 4) Pra-Pemrosesan Teks

Sebelum POS tagging, bersihkan teks dari karakter yang dapat menyebabkan error pada model NLP.

Fungsi `clean_text()` menghapus:
- Karakter *non-printable* (karakter kontrol)
- Karakter di luar rentang ASCII standard


In [12]:
def clean_text(text):
    """
    Membersihkan teks dari karakter non-printable dan non-ASCII
    agar kompatibel dengan model NLP berbahasa Indonesia.
    """
    text = str(text)
    # Hapus karakter non-printable (kategori 'C' = control characters)
    text = ''.join(char for char in text if unicodedata.category(char)[0] != 'C')
    # Hapus karakter non-ASCII
    text = re.sub(r'[^\x00-\x7F]+', '', text)
    return text

# Terapkan pembersihan ke kolom text_final
df_text['text_clean'] = df_text['text_final'].apply(clean_text)
print("Pembersihan teks selesai.")

# Tampilkan contoh
print("\nContoh teks setelah dibersihkan:")
print(df_text['text_clean'].iloc[0][:200])


Pembersihan teks selesai.

Contoh teks setelah dibersihkan:
kali kartu kredit cocok anak muda update alam pakai kartu terima kasih bimbing sangat baik


## 5) Deteksi Bahasa dengan langdetect

Sebelum POS tagging, deteksi bahasa dari tiap teks untuk memastikan model yang tepat digunakan. Model POS Stanza bersifat bahasa-spesifik (`id` untuk Bahasa Indonesia).


In [13]:
def detect_language(text):
    """
    Mendeteksi bahasa teks menggunakan langdetect.
    Return kode bahasa (misal: 'id', 'en') atau 'unknown'.
    """
    try:
        return detect(text)
    except LangDetectException:
        return 'unknown'


print("Deteksi bahasa pada 10 data pertama:")
print(f"{'No.':<5} {'Kode':<8} {'Cuplikan Teks'}")
print('-' * 65)

for idx, row in df_text.iterrows():
    lang_code = detect_language(row['text_clean'])
    cuplikan = row['text_clean'][:40] + '...'
    print(f"{idx+1:<5} {lang_code:<8} {cuplikan}")
    if idx == 9:
        break


Deteksi bahasa pada 10 data pertama:
No.   Kode     Cuplikan Teks
-----------------------------------------------------------------
1     id       kali kartu kredit cocok anak muda update...
2     id       hasil layan cs nama sangat sopan jelas s...
3     id       cepat aman...
4     hu       cs baik...
5     hr       jelas baik...
6     id       trimakasih ulfa sangat trimakasih tuju...
7     id       mantep gampang aktif fariz ramah enak aj...
8     id       alhamdulillah layan info cepat limit...
9     id       kartu kridit isi saldo dipake kecuali ti...
10    id       aktifasi cepat cs ramah...


## 6) POS Tagging pada Satu Dokumen

Demonstrasi POS tagging pada dokumen pertama dalam dataset. Setiap token akan diberi label kelas kata menggunakan tag Universal POS (UPOS) dari Stanza.

| Tag   | Kelas Kata             |
|-------|------------------------|
| NOUN  | Kata Benda             |
| VERB  | Kata Kerja             |
| ADJ   | Kata Sifat             |
| ADV   | Kata Keterangan        |
| PRON  | Kata Ganti             |
| DET   | Determiner             |
| ADP   | Kata Depan (Preposisi) |
| NUM   | Numeralia              |
| PUNCT | Tanda Baca             |
| PROPN | Kata Nama Diri         |


In [14]:
# Ambil dokumen pertama
doc_sample = df_text['text_clean'].iloc[0]
print("Teks sample:")
print(doc_sample[:200])
print("\n" + '='*50)


Teks sample:
kali kartu kredit cocok anak muda update alam pakai kartu terima kasih bimbing sangat baik



In [15]:
# POS Tagging dengan Stanza
doc = nlp(doc_sample)

upos_keterangan = {
    'NOUN': 'Kata Benda',
    'VERB': 'Kata Kerja',
    'ADJ': 'Kata Sifat',
    'ADV': 'Kata Keterangan',
    'PRON': 'Kata Ganti',
    'DET': 'Determiner',
    'ADP': 'Kata Depan (Preposisi)',
    'NUM': 'Numeralia',
    'PUNCT': 'Tanda Baca',
    'PROPN': 'Kata Nama Diri',
    'CCONJ': 'Konjungsi Koordinatif',
    'SCONJ': 'Konjungsi Subordinatif',
    'PART': 'Partikel',
    'SYM': 'Simbol',
    'X': 'Lain-lain',
}

print(f"{'Kata':<20} {'POS Tag':<10} {'Keterangan'}")
print('-' * 50)
for sentence in doc.sentences:
    for word in sentence.words:
        ket = upos_keterangan.get(word.upos, '')
        print(f"{word.text:<20} {word.upos:<10} {ket}")


Kata                 POS Tag    Keterangan
--------------------------------------------------
kali                 NOUN       Kata Benda
kartu                NOUN       Kata Benda
kredit               NOUN       Kata Benda
cocok                ADJ        Kata Sifat
anak                 NOUN       Kata Benda
muda                 ADJ        Kata Sifat
update               NOUN       Kata Benda
alam                 NOUN       Kata Benda
pakai                VERB       Kata Kerja
kartu                NOUN       Kata Benda
terima               NOUN       Kata Benda
kasih                NOUN       Kata Benda
bimbing              VERB       Kata Kerja
sangat               ADV        Kata Keterangan
baik                 ADJ        Kata Sifat


## 7) POS Tagging pada Seluruh Dataset

Terapkan POS tagging ke seluruh teks dalam dataset. Proses ini dapat memakan waktu tergantung ukuran dataset.

In [ ]:
all_pos_tags = []  # List untuk menampung hasil POS tags semua dokumen

for idx, text in df_text['text_clean'].items():
    try:
        doc = nlp(text)
        pos_tags = [(word.text, word.upos) for sent in doc.sentences for word in sent.words]
        all_pos_tags.append(pos_tags)
    except Exception as e:
        print(f"Error pada baris {idx}: {str(e)[:80]}")
        all_pos_tags.append([])

print(f"POS tagging selesai untuk {len(all_pos_tags)} dokumen.")

# Tampilkan contoh hasil 3 dokumen pertama
print("\nContoh hasil POS tags (3 dokumen pertama):")
for i, tags in enumerate(all_pos_tags[:3]):
    print(f"  Dokumen {i+1} (5 token pertama): {tags[:5]}")


## 8) Analisis Distribusi POS Tag

Setelah mendapatkan semua POS tags, hitung:
- Jumlah total token per kelas kata
- Jumlah token unik per kelas kata

Ini memberikan gambaran komposisi linguistik dari keseluruhan dataset.

In [ ]:
# Ratakan semua POS tags menjadi satu list
flat_pos_tags = [item for sublist in all_pos_tags for item in sublist]

# Buat DataFrame dari hasil POS tagging
df_word_pos = pd.DataFrame(flat_pos_tags, columns=['Word', 'POS Tag'])

# Hitung frekuensi setiap POS tag
tag_counts = df_word_pos['POS Tag'].value_counts()

# Jumlah token unik per POS tag
unique_tokens_per_tag = df_word_pos.groupby('POS Tag')['Word'].nunique()

# Gabungkan ke dalam summary DataFrame
summary_df = pd.DataFrame({
    'POS Tag'      : tag_counts.index,
    'Total Token'  : tag_counts.values,
    'Token Unik'   : unique_tokens_per_tag
}).reset_index(drop=True)

summary_df = summary_df.sort_values('Total Token', ascending=False).reset_index(drop=True)

print(f"Total token keseluruhan: {len(flat_pos_tags):,}")
print(f"Jumlah tag unik        : {len(summary_df)}")
print("\nDistribusi POS Tag:")
print(summary_df.to_string(index=False))

# Simpan ke CSV
summary_df.to_csv('pos_tags_summary.csv', index=False)
print("\nHasil disimpan ke 'pos_tags_summary.csv'")

## 9) Token Paling Sering per Kelas Kata

Tampilkan 15 kombinasi kata + POS tag yang paling sering muncul dalam seluruh dataset. Ini membantu memahami kata-kata kunci yang dominan dalam ulasan Honest Review.


In [ ]:
# Hitung frekuensi setiap pasangan (Word, POS Tag)
top_entities = (
    df_word_pos
    .groupby(['Word', 'POS Tag'])
    .size()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={0: 'Frekuensi'})
)

print("Top 15 Token berdasarkan Frekuensi:")
print(top_entities.head(15).to_string(index=False))

# Simpan ke CSV
top_entities.to_csv('pos_top_tokens.csv', index=False)
print("\nHasil disimpan ke 'pos_top_tokens.csv'")

## 10) Visualisasi Distribusi POS Tag

Tampilkan distribusi frekuensi POS tag dalam bentuk bar chart untuk mendapatkan gambaran yang lebih intuitif.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.bar(summary_df['POS Tag'], summary_df['Total Token'], color='steelblue', alpha=0.85, edgecolor='white')
plt.title('Distribusi POS Tag — Ulasan Honest Review', fontsize=13)
plt.xlabel('POS Tag')
plt.ylabel('Total Token')
plt.xticks(rotation=30, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()
